# Qiskit Fall Fest 2026, Challenge 5: Transpile golf

Opens Fri, Nov 20, 2026 at 6:00 PM. Closes Sun, Nov 22, 2026 at 11:59 PM. Both times are Clemson time (US Eastern).

**The task.** Fit a given circuit onto an IBM processor's layout with the fewest two-qubit gates, then run it there with error mitigation.

Start from Friday's notebook. Its Lab 2 transpiled one circuit forty times and kept the version with the fewest two-qubit gates. Its Lab 3 ran the winner twice, without and with error mitigation. Error mitigation is the name for techniques that correct an average result for the errors of a processor. This notebook gives you the circuit of the challenge, a check that says whether a circuit of yours counts, and the line of text that you turn in.

How to use this notebook:

- Run the three code cells of the Setup section first. Keep `HARDWARE = "ibm"`: the second option, qBraid, has no part in this challenge.
- Read "The task in detail". It says what counts and how a circuit is scored.
- Run the cells under "The tools". Their code is complete.
- Do your own work under "Your attempt". The search for a short circuit needs no account and no quantum computer. Only the run at the end needs a real one.
- Turn in the line that the notebook prints. The steps are under "Turn it in".

## Setup

Run the next three code cells in order, every time you open this notebook.

**Where your circuits run.** The first cell holds three switches.

- `USE_HARDWARE = False` keeps everything on the computer that runs this notebook (your laptop, or the server of Google Colab or qBraid Lab, two websites that run notebooks in your browser). Circuits run on a simulator, a program that imitates a quantum computer. No account is needed. Use this setting during the lab.
- `USE_HARDWARE = True` with `HARDWARE = "ibm"` sends the hardware cells to a real IBM quantum processor through your own IBM Quantum account. A hardware cell is a cell that can send work to a real quantum computer, and a processor is the chip that holds its qubits. This is the first choice for real hardware.
- `USE_HARDWARE = True` with `HARDWARE = "qbraid"` is the second option. qBraid is a service that gives access to quantum computers from several makers. The cells marked "second option" then use your qBraid account to reach the other quantum computers qBraid offers. qBraid calls each of them a device, and `QBRAID_DEVICE` holds the id of the device you pick there.

After you change a switch, run the switches cell below and the setup cell after it again.

In [ ]:
USE_HARDWARE = False      # True sends jobs to a real quantum computer
HARDWARE = "ibm"          # "ibm" (first choice) or "qbraid" (second option)
QBRAID_DEVICE = "PASTE_DEVICE_ID"

The next cell checks that the tested versions of the packages are installed. If one is missing or has a different version, the cell installs it and then stops with one line that asks you to restart the kernel. The kernel is the Python process that runs the cells of this notebook. After the restart, run the notebook again from the top.

The cell also defines `connect()`, the one helper that every hardware cell calls. `connect("ibm")` gives the lab code a service, the object you ask for a backend. A backend is the machine, real or simulated, that runs a circuit. With `USE_HARDWARE = False` the service hands out a simulated copy of the IBM processor `ibm_fez`, built from a saved record of that machine's error rates. Qiskit names this copy `fake_fez`.

In [ ]:
# Setup. Run this cell every time, right after the switches cell above. It checks the packages, then defines connect().
import sys
from importlib.metadata import PackageNotFoundError, version

PINNED = {"qiskit": "2.6.2", "qiskit-ibm-runtime": "0.52.1", "qiskit-aer": "0.18.2"}
if str(HARDWARE).strip().lower() == "qbraid":
    PINNED["qbraid"] = "0.13.0"                        # needed only for the second option
EXTRA = ["matplotlib", "pylatexenc", "scikit-learn"]   # any version will do


def _found(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return None


def _missing():
    wanted = [f"{name}=={pin}" for name, pin in PINNED.items() if _found(name) != pin]
    return wanted + [name for name in EXTRA if _found(name) is None]


class StopHere(Exception):
    """Stops the run with one plain line in place of a long error report."""

    def _render_traceback_(self):
        return [str(self)]


if sys.version_info < (3, 11):
    raise StopHere("This notebook needs Python 3.11 or newer. This kernel runs Python "
                   + sys.version.split()[0] + ". Choose a newer Python, then run this cell again.")

if _missing():
    _spec = " ".join(_missing())
    %pip install -q {_spec}
    if _missing():
        raise StopHere("Some packages could not be installed: " + ", ".join(_missing())
                       + ". Read the message above this line, or ask an organizer.")
    raise StopHere("Packages were installed. Restart the kernel now, then run the notebook "
                   "again from the top. (Jupyter and qBraid Lab: Kernel menu, Restart Kernel. "
                   "Google Colab: Runtime menu, Restart session.)")

import os
from getpass import getpass

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime.accounts import AccountNotFoundError
from qiskit_ibm_runtime.fake_provider import FakeFez

class SimulatedBackend(FakeFez):
    """A simulated copy of the IBM processor ibm_fez. It runs where this notebook runs, and nothing is sent."""

    def __repr__(self):
        return f"<{type(self).__name__}('{self.name}')>"


SIMULATED_BACKEND = SimulatedBackend()


class SimulatedService:
    """Stands in for QiskitRuntimeService when nothing is sent to IBM.

    It answers the same calls and always hands out the simulated processor.
    """

    def backends(self, **filters):
        return [SIMULATED_BACKEND]

    def backend(self, name=None, **filters):
        return SIMULATED_BACKEND

    def least_busy(self, **filters):
        return SIMULATED_BACKEND


def connect(account):
    """Return the entry point for one account. Every hardware cell calls this.

    connect("ibm") returns a service. service.least_busy(...) then gives a backend.
        USE_HARDWARE = True and HARDWARE = "ibm": your own IBM Quantum account.
        Any other setting: a simulated IBM processor. No account is used.
    connect("qbraid") returns a qBraid provider, or None when that route is off.
        USE_HARDWARE = True and HARDWARE = "qbraid": your qBraid account.
        Any other setting: None. Nothing is sent.
    """
    route = str(HARDWARE).strip().lower()
    if not isinstance(USE_HARDWARE, bool) or route not in ("ibm", "qbraid"):
        raise ValueError("Check the first code cell: USE_HARDWARE must be True or False, "
                         'and HARDWARE must be "ibm" or "qbraid".')
    if account not in ("ibm", "qbraid"):
        raise ValueError('Call connect("ibm") or connect("qbraid").')
    live = USE_HARDWARE and route == account
    why = "USE_HARDWARE is False" if not USE_HARDWARE else f'HARDWARE is "{route}"'

    if account == "ibm":
        if not live:
            print(f"Simulator: this cell runs on {SIMULATED_BACKEND.name}, a simulated IBM "
                  f"processor ({why}). No account is used and nothing is sent.")
            return SimulatedService()
        try:
            service = QiskitRuntimeService()
        except AccountNotFoundError:
            print("No IBM Quantum account is saved on this computer. Paste your API key and "
                  "your instance CRN in the boxes. They stay hidden. They are saved in your "
                  "home folder, never in this notebook.")
            QiskitRuntimeService.save_account(
                token=getpass("IBM Quantum API key: "),
                instance=getpass("Instance CRN (press Enter to skip): ") or None,
                set_as_default=True, overwrite=True)
            service = QiskitRuntimeService()
        print("Real hardware: this cell uses your own IBM Quantum account and your quantum time.")
        return service

    if not live:
        print(f"Skipped: this cell uses a qBraid account, the second option ({why}). "
              "Nothing is sent.")
        return None
    from qbraid.runtime import QbraidProvider
    if not (os.environ.get("QBRAID_API_KEY")
            or os.path.exists(os.path.expanduser("~/.qbraid/qbraidrc"))):
        print("No qBraid API key is saved on this computer. Paste it in the box. It stays "
              "hidden and is kept only until this kernel stops.")
        os.environ["QBRAID_API_KEY"] = getpass("qBraid API key: ")
    if QBRAID_DEVICE == "PASTE_DEVICE_ID":
        print('QBRAID_DEVICE is still "PASTE_DEVICE_ID". Copy the id of an online device from '
              "the list this cell prints into the first code cell, run that cell, then run "
              "this cell again.")
    print("Real hardware, second option: this cell uses your qBraid account. Device:", QBRAID_DEVICE)
    return QbraidProvider()

The setup check prints what is installed and where circuits will run. If the versions match the tested ones, you are ready.

In [ ]:
# Setup check: what is installed, and where circuits will run.
import qiskit, qiskit_aer, qiskit_ibm_runtime   # an error here means an install went wrong

print("Python", sys.version.split()[0])
for _name, _pin in PINNED.items():
    print(f"{_name} {_found(_name)}  (tested with {_pin})")
print("USE_HARDWARE =", USE_HARDWARE, "| HARDWARE =", repr(HARDWARE),
      "| QBRAID_DEVICE =", repr(QBRAID_DEVICE))
print("Simulated IBM processor:", SIMULATED_BACKEND.name, "with",
      SIMULATED_BACKEND.num_qubits, "qubits")

## The task in detail

### The given circuit

The circuit is Tuesday's QAOA circuit, the circuit of the quantum approximate optimization algorithm: the graph of Tuesday's lab, with 12 nodes and 16 edges, one layer, and Tuesday's angles $\beta = 1.20$ and $\gamma = 0.32$. It has 12 H gates, 16 RZZ gates with the angle $2\gamma = 0.64$ and 12 RX gates with the angle $2\beta = 2.40$, and no measurements. `given_plain`, defined under "The tools", holds these gates in order. An RZZ gate acts on two qubits, and the circuit has one for each edge of the graph.

### The layout

Three IBM processors of the Open Plan of IBM Quantum, the plan you do not pay for, share one layout: 156 qubits and 176 couplers. Test jobs for this event ran on two of them, ibm_fez and ibm_marrakesh, on Nov 13, 2026. IBM's documentation names the third, ibm_kingston (IBM Quantum documentation, "IBM Quantum Compute Service changelog", https://quantum.cloud.ibm.com/docs/en/guides/changelog-quantum-compute-service, read on Nov 15, 2026). A coupler is the connection between two neighboring qubits, and a two-qubit gate works only across a coupler. The simulated processor in this notebook has the same layout, so you can search without an account.

The given circuit does not fit this layout as it is. Its gates are not the processor's gates, and its 16 edges do not lie on 16 couplers. To transpile is to rewrite a circuit in the gates of one processor and to place it on that processor's qubits. Wherever two qubits that must interact are not neighbors, the transpiler adds two-qubit gates to bring them together. Those extra gates are what this challenge is about, because every two-qubit gate adds error.

### What you turn in

A circuit of your own on the 156 qubits, and a list of 12 physical qubits. A physical qubit is one of the numbered qubits of the chip. Entry $k$ of the list is the physical qubit that holds qubit $k$ of the given circuit when your circuit ends.

The circuit counts when it passes two tests.

- **It fits.** Every instruction is one of four: `cz` on two qubits that a coupler joins, or `rz`, `sx` or `x` on one qubit. Nothing else is allowed: no other gate, no measurement, no reset, no barrier and no delay. The circuit touches at most 20 qubits and has at most 1,000 instructions. These two limits keep the check fast. The circuit that Friday's loop finds has about 250 instructions.
- **It prepares the same state.** Starting from all zeros, your circuit leaves the twelve qubits of your list in the state that the given circuit prepares, and every other qubit it touches in 0. The test is a fidelity of at least 0.9999. The fidelity of two states is the squared size of their overlap. It is 1 when they are the same state and 0 when they have nothing in common. A factor of size 1 in front of the whole state, called a global phase, does not matter.

Any circuit that passes both tests counts, however you found it.

### Your score

The count is the number of `cz` gates in your circuit.

$$\text{score} = \max\left(0, \; \min\left(100, \; 100 - 4 \times (\text{gates} - 48)\right)\right)$$

48 gates or fewer score 100. Every gate above 48 costs 4 points: 53 gates score 80, 58 score 60, and 73 or more score 0. You may turn in a line as often as you like, and your valid line with the fewest gates counts. Below 48 gates the score stays at 100. The Challenge 5 board lists people by points, the score plus any bonus. Among equal points, the person with fewer gates is listed first. How the event uses the scores is explained on the event page.

### The run

The second half of the task is to run your circuit on a real IBM processor, as in Friday's Lab 3: once without error mitigation and once with it. Both runs measure the average cut value of Tuesday's graph in the state your circuit prepares. Without any error that value is 10.937.

- The run does not change your score. A line without a run is valid.
- The event page says that the mitigated result breaks ties. Among people with the same points and the same number of gates, a line with a run comes before a line without one, then the mitigated value closer to 10.937 comes first, and after that the line turned in earlier.
- If you turn in several lines with your smallest number of gates, the first of them that has a run is the one that is compared. Running the same circuit again does not move you up.
- The run needs your own IBM Quantum account, because the layout is that of IBM's processors. The second option, qBraid, has no part in this challenge.
- The two jobs are created after the challenge opens and before it closes. The submission form closes at the same time, and a line, with or without a run, counts only if the form has it by Sun, Nov 22 at 11:59 PM, so leave time for the queue.

### What to expect

Friday's loop, unchanged, finds a circuit with 53 two-qubit gates for this layout, which scores 80. More seeds help a little. The full score takes an idea, and here is one sentence of help:

The sixteen RZZ gates commute, so they can be applied in any order, and the transpiler does not use that freedom.

## The tools

The code in this part is complete. Run each cell and read what it does. You will call these functions in "Your attempt".

The first cell builds the given circuit, with the two lines of Tuesday's Lab 2, and the quantity that the run measures.

- `given` is the given circuit with Tuesday's angles filled in.
- `cut_op` is the cut value of Tuesday's graph, written as an operator: every edge adds $\frac{1 - Z_i Z_j}{2}$, which is 1 when the two ends of the edge are on different sides and 0 when they are on the same side.
- `ideal` is the average cut value in the state that the given circuit prepares, computed exactly.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.circuit.library import qaoa_ansatz
from qiskit.quantum_info import SparsePauliOp, Statevector

n = 12                 # qubits of the given circuit, one for each node
TARGET = 48            # two-qubit gates that earn the full score

# Tuesday's lab graph: 12 nodes, 16 edges.
edges = [(0, 1), (0, 4), (1, 2), (1, 5), (2, 3), (3, 6), (3, 7), (4, 5),
         (4, 8), (5, 9), (6, 7), (6, 11), (7, 11), (8, 9), (9, 10), (10, 11)]
angles = [1.20, 0.32]  # Tuesday's angles: [beta, gamma]

cost = SparsePauliOp.from_sparse_list(
    [("ZZ", [i, j], 1.0) for i, j in edges],
    num_qubits=n)
given = qaoa_ansatz(cost, reps=1).assign_parameters(angles)

cut_op = SparsePauliOp.from_sparse_list(
    [("ZZ", [i, j], -0.5) for i, j in edges] + [("I", [0], len(edges) / 2)],
    num_qubits=n)
ideal = float(Statevector(given).expectation_value(cut_op).real)

print("gates of the given circuit:", dict(given.count_ops()))
print("average cut value without any error:", round(ideal, 3))

The second cell holds the layout and the two tests.

- `COUPLERS` is the set of the 176 couplers, read from the simulated processor.
- `count_gates(circuit)` is the count of the challenge: the number of `cz` gates.
- `fits(circuit)` is the first test. It returns `True` or `False` and, when the answer is `False`, the reason.
- `fidelity_with_given(circuit, qubits)` is the number behind the second test. It works out, without noise and without a quantum computer, the state that your circuit prepares and compares it with the state of the given circuit placed on the physical qubits of your list.
- `check(circuit, qubits)` runs both tests. It returns `True` or `False` and the reason. The organizers use the same function on the circuit in your line.

You do not need to follow the code of this cell to use it.

In [ ]:
from qiskit.circuit.library import CZGate, RZGate, SXGate, XGate
from qiskit.transpiler import generate_preset_pass_manager

CHIP_QUBITS = 156
MAX_TOUCHED = 20           # a circuit may touch at most this many qubits
MAX_INSTRUCTIONS = 1000    # and may have at most this many instructions
MIN_FIDELITY = 0.9999      # the smallest fidelity that counts as the same state
ALLOWED = {"cz": CZGate, "rz": RZGate, "sx": SXGate, "x": XGate}
COUPLERS = {tuple(sorted(pair)) for pair in SIMULATED_BACKEND.coupling_map.get_edges()}

# The given circuit written with plain H, RZZ and RX gates, for the comparison of states.
given_plain = generate_preset_pass_manager(optimization_level=0, basis_gates=["rzz", "rx", "h"]).run(given)


def count_gates(circuit):
    """The count of the challenge: the number of cz gates."""
    return sum(1 for inst in circuit.data if inst.operation.name == "cz")


def touched_qubits(circuit):
    """The qubits that at least one instruction of the circuit acts on."""
    return sorted({circuit.find_bit(q).index for inst in circuit.data for q in inst.qubits})


def fits(circuit):
    """Return (True, "") if the circuit fits the layout, else (False, the reason)."""
    if circuit.num_qubits != CHIP_QUBITS:
        return False, "the circuit has " + str(circuit.num_qubits) + " qubits, and the layout has " + str(CHIP_QUBITS)
    if circuit.num_clbits or circuit.parameters:
        return False, "the circuit must have no classical bits, no measurements and no open angles"
    if len(circuit.data) > MAX_INSTRUCTIONS:
        return False, "the circuit has " + str(len(circuit.data)) + " instructions, more than " + str(MAX_INSTRUCTIONS)
    for inst in circuit.data:
        name = inst.operation.name
        qubits = tuple(circuit.find_bit(q).index for q in inst.qubits)
        if name not in ALLOWED or not isinstance(inst.operation, ALLOWED[name]):
            return False, "the instruction '" + name + "' is not one of cz, rz, sx, x"
        if name == "cz" and tuple(sorted(qubits)) not in COUPLERS:
            return False, "cz on the qubits " + str(qubits) + ": no coupler joins these two"
    if len(touched_qubits(circuit)) > MAX_TOUCHED:
        return False, "the circuit touches " + str(len(touched_qubits(circuit))) + " qubits, more than " + str(MAX_TOUCHED)
    return True, ""


def fidelity_with_given(circuit, qubits):
    """The fidelity between the state the circuit prepares and the given state placed on these physical qubits."""
    qubits = [int(q) for q in qubits]
    touched = touched_qubits(circuit)
    if len(qubits) != given_plain.num_qubits or len(set(qubits)) != given_plain.num_qubits or not set(qubits) <= set(touched) or len(touched) > MAX_TOUCHED:
        return 0.0                                # the list needs twelve different qubits that the circuit acts on
    place = {q: k for k, q in enumerate(touched)}
    mine = QuantumCircuit(len(touched))           # the same gates on a small register
    for inst in circuit.data:
        mine.append(inst.operation, [place[circuit.find_bit(q).index] for q in inst.qubits])
    wanted = QuantumCircuit(len(touched))         # the given circuit on the positions of the list, the rest in 0
    wanted.compose(given_plain, qubits=[place[q] for q in qubits], inplace=True)
    overlap = Statevector(wanted).inner(Statevector(mine))
    return float(abs(overlap) ** 2)


def check(circuit, qubits):
    """Return (True, "") if the circuit counts for the challenge, else (False, the reason)."""
    ok, why = fits(circuit)
    if not ok:
        return False, why
    fidelity = fidelity_with_given(circuit, qubits)
    if not fidelity >= MIN_FIDELITY:
        return False, "the circuit does not prepare the state of the given circuit on these qubits (fidelity " + format(fidelity, ".4f") + ")"
    return True, ""


print("qubits of the layout:", CHIP_QUBITS, "| couplers:", len(COUPLERS))

The third cell writes the line you turn in. The line holds your whole circuit, so that the organizers can run the same check on it.

- `pack(circuit)` writes the circuit as text in OpenQASM 2, a standard text format for circuits, compresses that text, and writes the result in letters, digits, `-` and `_`.
- `unpack(text)` turns packed text back into a circuit.
- `score(gates)` is the score formula from above.
- `submission_line(...)` writes the line. Without a run it says `jobs=none`.

In [ ]:
import base64
import zlib

import qiskit
import qiskit_ibm_runtime
from qiskit import qasm2


def pack(circuit):
    """The circuit as packed text: OpenQASM 2, compressed, written in letters, digits, - and _."""
    return base64.urlsafe_b64encode(zlib.compress(qasm2.dumps(circuit).encode(), 9)).decode().rstrip("=")


def unpack(text):
    """The circuit that a packed text holds."""
    raw_bytes = base64.urlsafe_b64decode(text + "=" * (-len(text) % 4))
    return qasm2.loads(zlib.decompress(raw_bytes).decode(), custom_instructions=qasm2.LEGACY_CUSTOM_INSTRUCTIONS)


def score(gates):
    """The score for a valid circuit with this many cz gates."""
    return max(0, min(100, 100 - 4 * (gates - TARGET)))


def submission_line(circuit, qubits, created_utc, device="none", jobs=None, raw=None, mitigated=None):
    """The one line of text that you turn in. Leave out the last four entries for a line without a run."""
    with_run = jobs is not None
    return ("QFF26-C5 device=" + str(device) + " jobs=" + (",".join(jobs) if with_run else "none")
            + " gates=" + str(count_gates(circuit))
            + " raw=" + (format(raw, ".3f") if with_run else "none")
            + " mitigated=" + (format(mitigated, ".3f") if with_run else "none")
            + " qubits=" + ",".join(str(int(q)) for q in qubits) + " circuit=" + pack(circuit)
            + " utc=" + created_utc + " qiskit=" + qiskit.__version__ + " runtime=" + qiskit_ibm_runtime.__version__)

The last cell of this part tries the tools on one transpilation: Monday's pass manager (the Qiskit object that runs the steps of the transpiler one after another, built with `generate_preset_pass_manager`), optimization level 1, with seed 0. The optimization level, from 0 to 3, says how hard the transpiler works to shorten the circuit, and a seed fixes its random choices. `trial.layout.final_index_layout()` is the list of physical qubits: where the transpiler put each qubit of the given circuit by the end.

The cell also packs the circuit and unpacks it again, because that is the circuit the organizers will see.

In [ ]:
trial = generate_preset_pass_manager(optimization_level=1, backend=SIMULATED_BACKEND, seed_transpiler=0).run(given)
trial_qubits = trial.layout.final_index_layout()

print("gates after transpiling:", dict(trial.count_ops()))
print("physical qubits:", trial_qubits)
print("qubits touched:", len(touched_qubits(trial)), "| fidelity with the given state:", round(fidelity_with_given(trial, trial_qubits), 6))
print("check:", check(trial, trial_qubits))
print("two-qubit gates:", count_gates(trial), "| score:", score(count_gates(trial)))
print("packed text:", len(pack(trial)), "characters | check after unpacking:", check(unpack(pack(trial)), trial_qubits))

One transpilation at level 1 gives 68 two-qubit gates, which scores 20. The check passes, before and after packing. Where do 68 gates come from? Each of the 16 RZZ gates becomes two `cz` gates, which makes 32. The other 36 are routing: they move qubits that are not neighbors next to each other.

## Your attempt

From here on the work is yours.

The first cell below is Friday's golf loop with two changes. It transpiles `given`, and it keeps a circuit only if it passes the check. The second change matters: some transpiler settings return a circuit that is short because it is no longer the same circuit. A low `approximation_degree` is one of them. The loop calls its count `gates`, not `n` as Friday's loop does. Do not give your own variables the names that the tools use, such as `given`, `given_plain`, `check` and `score`.

The loop ends with the two things you turn in: `my_circuit` and `my_qubits`. Change the search, or build `my_circuit` and `my_qubits` in any other way you like.

What is left to you:

- **How to search further.** Forty transpilations are a start. `golf_backend` is the simulated processor. It has the layout of the real ones, so the count you find here holds there.
- **What to do with the sentence of help.** It is in "What to expect".
- **Whether to build the state in another way.** Any circuit that passes the check counts. For a circuit that did not come from the transpiler, write the list `my_qubits` yourself.
- **Which circuit to run.** Two circuits with the same count can sit on different qubits, and the qubits of a chip differ in quality. Friday's Lab 1 showed how to read the error rates of a backend.

In [ ]:
golf_backend = SIMULATED_BACKEND       # the layout is the same on the real processors

best = None
for level in range(4):
    for seed in range(10):
        pm = generate_preset_pass_manager(
            optimization_level=level,
            backend=golf_backend,
            seed_transpiler=seed)
        isa = pm.run(given)
        gates = count_gates(isa)
        ok, why = check(isa, isa.layout.final_index_layout())
        if ok and (best is None or gates < best[0]):
            best = (gates, level, seed, isa)
print(best[:3])

my_circuit = best[3]
my_qubits = my_circuit.layout.final_index_layout()

The loop prints `(53, 3, 7)`: 53 two-qubit gates, found at optimization level 3 with seed 7.

The next cell checks what you will turn in and prints your line without a run. That line is valid as it is. The cell checks the circuit after packing and unpacking it, which is exactly what the organizers do.

In [ ]:
from datetime import datetime, timezone

sent = unpack(pack(my_circuit))        # the circuit as the organizers will see it
ok, why = check(sent, my_qubits)
if not ok:
    print("This circuit does not count: " + why)
else:
    now = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
    print("Valid circuit with", count_gates(sent), "two-qubit gates on", len(touched_qubits(sent)), "qubits. Score:", score(count_gates(sent)), "points.")
    print("Your line without a run:")
    print(submission_line(sent, my_qubits, now))

### Run it on a real processor

The next cell runs your circuit as Friday's Lab 3 did, with the Estimator, the Qiskit tool that returns the average value of a quantity such as `cut_op`. It sends one job without mitigation, then one job with `resilience_level = 2` and dynamical decoupling. `resilience_level = 2` is the setting that switches the mitigation on. Dynamical decoupling adds pulses to qubits that are waiting, so that they keep their state. Both jobs measure `cut_op`. The line `cut_op.apply_layout(my_qubits, ...)` rewrites the operator for the physical qubits of your list.

With `USE_HARDWARE = False` the cell is a practice run on a lighter model of the simulated processor, built as in Friday's Lab 3. It takes about a minute, and almost all of that is the mitigated job. With `USE_HARDWARE = True` and `HARDWARE = "ibm"` it sends the two jobs to an IBM processor of your account that can run your circuit.

Before anything is sent, the cell runs `check` on your circuit, so that no quantum time, the time a processor spends on your jobs, goes to a circuit that does not count. Then it chooses the processor. A real processor can have a coupler or a qubit switched off, and its record then reports no gate there, or a gate with an error of 1. So the cell goes through the processors of your account, the one with the fewest jobs waiting first, and takes the first one that has the layout of the challenge and every gate your circuit uses. It prints which ones it passed over and why. If none can run your circuit, the cell stops and says so. Choose another circuit then. To name a processor yourself, set `MY_BACKEND` at the top of the code cell below, for example `MY_BACKEND = "ibm_fez"`. The cell then tries only that one. On the simulator the setting changes nothing, because the simulator always uses fake_fez.

On Nov 14, 2026, a circuit of this challenge with 42 two-qubit gates used 3 seconds of quantum time for the first job and 7 seconds for the second on the IBM processor ibm_fez. Its values were 10.52 without mitigation and 10.85 with it. The Open Plan of IBM Quantum, the plan you do not pay for, gives up to 10 minutes of quantum time in every 28-day window (IBM Quantum documentation, "Plans overview", https://quantum.cloud.ibm.com/docs/en/ibm-quantum-platform/plans-overview, read on Nov 13, 2026).

**If your jobs are stuck in the queue.** Interrupt the kernel (Jupyter and qBraid Lab: Kernel menu, Interrupt Kernel. Google Colab: Runtime menu, Interrupt execution). Interrupting stops the waiting, not the jobs: they stay in IBM's queue. A challenge notebook has no stored results to fall back on. When their turn comes the two jobs still run and use quantum time. So cancel them before you run the cell again: run `mit.cancel()` in a new cell and `raw.cancel()` in another. A job that has already finished cannot be cancelled, and its cell may then stop with an error; that job needs nothing more. Your line without a run stays valid.

**Hardware cell (IBM Quantum).** With `USE_HARDWARE = True` and `HARDWARE = "ibm"` this cell uses your own IBM Quantum account. With any other setting it runs on a simulated IBM processor and uses no account.

In [ ]:
MY_BACKEND = None      # None: the least busy processor that can run your circuit. Or a name in quotes, such as "ibm_fez"

import warnings

from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error
from qiskit_ibm_runtime import EstimatorV2

ok, why = check(my_circuit, my_qubits)      # a circuit that does not count is not sent
assert ok, "This circuit does not count, so nothing is sent: " + why


def trouble_on(backend):
    """Why this processor cannot run my_circuit today: an empty list if it can."""
    if backend.num_qubits != CHIP_QUBITS:
        return ["it has " + str(backend.num_qubits) + " qubits, and the challenge needs the layout with 156"]
    record = backend.target                # the calibration record of the processor
    trouble = []
    for inst in my_circuit.data:
        place = tuple(my_circuit.find_bit(q).index for q in inst.qubits)
        name = inst.operation.name
        if not record.instruction_supported(name, place):
            trouble.append(name + " on " + str(place) + " does not exist on this processor today")
        elif name == "cz" and record["cz"][place] is not None and (record["cz"][place].error or 0) >= 0.5:
            trouble.append("the coupler " + str(place) + " is switched off today")
    return sorted(set(trouble))[:5]


service = connect("ibm")
on_simulator = type(service).__name__ == "SimulatedService"      # True: the simulated processor, and nothing is sent
if MY_BACKEND:
    candidates = [service.backend(MY_BACKEND)]                    # only the processor you named
else:
    candidates = service.backends(operational=True, simulator=False)
    if len(candidates) > 1:
        candidates = sorted(candidates, key=lambda b: b.status().pending_jobs)      # the fewest jobs waiting first

# Before any quantum time is spent: take the first processor that has every gate that the circuit uses.
backend, passed = None, []
for candidate in candidates:
    trouble = trouble_on(candidate)
    if not trouble:
        backend = candidate
        break
    passed.append(candidate.name + ": " + "; ".join(trouble))
    print("passed over " + passed[-1])
assert backend is not None, "This circuit cannot run on " + (" | ".join(passed) or "any processor: your account lists none")
print("backend:", backend.name)
record = backend.target                    # the calibration record of the processor

run_backend = backend
if on_simulator:
    # Simulated processor only, as in Friday's Lab 3: a lighter model with errors on the qubits of this circuit.
    used = touched_qubits(my_circuit)
    noise = NoiseModel(basis_gates=list(record.operation_names))
    for q in used:
        ro = record["measure"][(q,)].error                      # the readout error of this qubit
        noise.add_readout_error([[1 - ro, ro], [ro, 1 - ro]], [q])
        noise.add_quantum_error(depolarizing_error(record["sx"][(q,)].error, 1), ["sx", "x"], [q])
    for pair, prop in record["cz"].items():
        if set(pair) <= set(used):                              # the two-qubit gates among these qubits
            noise.add_quantum_error(depolarizing_error(prop.error, 2), "cz", pair)
    run_backend = AerSimulator.from_backend(backend, noise_model=noise, method="matrix_product_state")

obs_isa = cut_op.apply_layout(my_qubits, num_qubits=CHIP_QUBITS)
warnings.filterwarnings("ignore", message="No positive, finite standard errors")   # a note of the fit to zero noise, not an error
est = EstimatorV2(mode=run_backend)

est.options.resilience_level = 0    # raw
raw = est.run([(my_circuit, obs_isa)])

est.options.resilience_level = 2    # mitigated
dd = est.options.dynamical_decoupling
dd.enable = True
mit = est.run([(my_circuit, obs_isa)])

raw_now = float(raw.result()[0].data.evs)
mit_now = float(mit.result()[0].data.evs)
raw_value, mit_value = raw_now, mit_now      # kept only together with the jobs below, once both results are in
run_circuit, run_qubits = my_circuit, [int(q) for q in my_qubits]      # the circuit these two jobs ran: the next cell uses it
run_device, run_practice = backend.name, on_simulator
run_raw, run_mit = raw, mit              # the two jobs of this run
print("raw:", round(raw_value, 3), "| mitigated:", round(mit_value, 3), "| ideal:", round(ideal, 3))

The next cell prints your line with the run. It prints the line only for a run on a real processor. The line holds the circuit that the two jobs ran, whatever `my_circuit` holds by now. If you find a better circuit afterwards, run the cell of the run again before you print this line.

In [ ]:
from datetime import timezone

sent = unpack(pack(run_circuit))       # the circuit of the run, as the organizers will see it
ok, why = check(sent, run_qubits)
if run_practice:
    print("This was a practice run on the simulated processor. It is not part of a submission.")
    print("Your line without a run, printed further up, is valid on its own.")
elif not ok:
    print("This circuit does not count: " + why)
else:
    created = run_mit.creation_date.astimezone(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
    print("Valid circuit with", count_gates(sent), "two-qubit gates. Score:", score(count_gates(sent)), "points. Your line with the run:")
    print(submission_line(sent, run_qubits, created, run_device, [run_raw.job_id(), run_mit.job_id()], raw_value, mit_value))

## Turn it in

1. Copy the whole line that starts with `QFF26-C5`. It is long, about a thousand characters, because it holds your circuit. Do not change anything in it.
2. Open the submission form: [https://forms.gle/LdGAKjrSbNfeYNRT6](https://forms.gle/LdGAKjrSbNfeYNRT6)
3. Enter the email you registered with and a leaderboard name, paste the line, and send. The name is shown on the leaderboards of the event page. Type the same name every time: the boards keep the name you typed first in any of the five forms, and only an organizer can change it.

You can turn in a new line whenever you have a circuit with fewer gates. Your valid line with the fewest gates counts. A line without a run counts for the score. The first ten people to reach 100 get a bonus, in the order in which the form receives their first line worth 100. So turn in your line without a run as soon as it scores 100. To have your run compared as well, turn in the line with the run later.

## If something breaks

| What you see | What to do |
|---|---|
| The setup cell stops and asks for a restart | Restart the kernel, then run the notebook again from the top. It installed the tested packages |
| "the instruction '...' is not one of cz, rz, sx, x" | The circuit still has a barrier or a gate that the processor does not have. Transpile it with `backend=golf_backend` |
| "the circuit has 12 qubits, and the layout has 156" | The circuit was not transpiled for the layout. Transpile it with `backend=golf_backend`, or build it on `QuantumCircuit(156)` |
| "the circuit must have no classical bits, no measurements and no open angles" | Take out `measure_all()`, and fill in the angles with `assign_parameters(angles)` |
| `NameError: name '...' is not defined` | Run the cells in order, from the top. "Run it on a real processor" needs the cells above it in "Your attempt", and those need "The tools" |
| "no coupler joins these two" | A `cz` gate sits on two qubits that are not neighbors on the layout |
| "the circuit does not prepare the state of the given circuit" | The circuit was changed in a way that changes its state, or `my_qubits` is not the list that belongs to it. For a transpiled circuit the list is `my_circuit.layout.final_index_layout()` |
| `TypeError: 'NoneType' object is not subscriptable` in the loop | No circuit passed the check, so `best` is still empty. Print `why` inside the loop to see the reason |
| "This circuit does not count, so nothing is sent: ..." | The check refuses `my_circuit` with `my_qubits`, for the reason after the colon. Nothing was sent. Fix the circuit or the list, then run the cell again |
| "This circuit cannot run on ..." | No processor that the cell tried can run your circuit today. After each name the message gives the reasons: a coupler or a qubit that your circuit uses is missing or switched off, or the processor does not have the layout with 156 qubits. Nothing was sent. Choose another circuit, or name another processor in `MY_BACKEND` at the top of the cell of the run |
| Practice run: "the coupler ... is switched off today" on fake_fez | The saved record of the simulated processor marks 7 couplers as switched off. Your circuit still counts. For the practice run, move it to other qubits. On the simulator, `MY_BACKEND` changes nothing |
| The mitigated value is farther from 10.937 than the raw value | That can happen. Mitigation works on average, and a single run scatters |
| The jobs stay queued | Interrupt the kernel and cancel the two jobs: run `mit.cancel()` in a new cell and `raw.cancel()` in another. A finished job cannot be cancelled, and its cell may then stop with an error. Then run the cell of the run again later. Jobs that are not cancelled still run and use quantum time |
| The notebook asks for an API key | No IBM Quantum account is saved on this computer. Monday's notebook, section "Accounts", shows how to get the key |
| IBM rejects your key (an error that mentions the API key, the token or 401) | A wrong key is saved. Open Monday's notebook, set `REPLACE_SAVED_KEY = True` in the cell under "Save your IBM credentials once", run it with the right key, then come back here. Never type the key into a code cell |

## Where to go from here

- Every challenge stays open until Sun, Nov 22 at 11:59 PM, so there is time to come back with a better circuit.
- Stuck? Ask in Discord (discord.gg/FBNzRcxMr), also if a score looks wrong.